# Traffic CCTV — thử checkpoint đã fine-tune trên giao thông thực tế (Kaggle)

**Mục tiêu:** detect từng xe máy/xe hai bánh trên **một ảnh** do bạn cung cấp qua `/kaggle/input/...`, vẽ bounding box, đếm xe, xuất CSV. **Chỉ inference, KHÔNG train.**

### Nguồn checkpoint (phân biệt rõ)

- **BMD-45 — AIM@IISc (Ấn Độ)**: có weights YOLOv12-X (`best.pt` ~356 MB) và YOLOv12-S (`best.pt` ~18.9 MB) công bố công khai tại [Hugging Face](https://huggingface.co/iisc-aim/BMD-45/tree/main/weights). Đã fine-tune trên dữ liệu giao thông, gồm class **Two-wheeler** (xe máy/scooter, thường bao cả người lái trong bbox). **Không phải weights của paper Việt Nam**.
- **Paper Việt Nam (Vo, Nguyen & Tran, CMC 2026, DOI: 10.32604/cmc.2026.078756)**: nhóm tác giả dùng YOLO11m fine-tuned, nhưng chưa xác minh được liên kết tải weights công khai. Chế độ `paper_local` **chỉ chạy khi bạn đã có checkpoint từ tác giả**, không thay lén bằng COCO.

⚠️ mAP của các paper/dataset **không** đo trực tiếp độ chính xác trên ảnh của bạn. Kết quả số bbox phát hiện không đồng nghĩa ground-truth. `YOLOv12-X` không được đảm bảo hơn `S` nếu chưa thử cả hai trên chính dữ liệu camera của bạn.

**Trên Kaggle:** Add Input dataset ảnh → Settings → Accelerator = GPU → bật Internet để tự tải checkpoint Hugging Face lần đầu. Output lưu ở `/kaggle/working/traffic_detection_output/`.


In [ ]:
%pip install -q -U ultralytics huggingface_hub pillow pandas matplotlib


## 1) Cấu hình duy nhất cần sửa

Thay `IMAGE_PATH` bằng đường dẫn **ảnh bất kỳ** của bạn trong Kaggle. `MODEL_CHOICE`:

- `"bmd_x"`: checkpoint BMD-45 YOLOv12-X (mặc định, nặng, ưu tiên thử độ chính xác)
- `"bmd_s"`: checkpoint BMD-45 YOLOv12-S (nhẹ hơn)
- `"paper_local"`: dùng chính `best.pt` do nhóm tác giả **paper Việt Nam** gửi cho bạn
- `"custom_local"`: dùng bất kỳ checkpoint Ultralytics `.pt` hợp lệ nào

Để đổi ảnh, chỉ sửa `IMAGE_PATH` và chạy lại các cell bên dưới (không cần cài lại).

In [ ]:
# >>> THAY ẢNH CỦA BẠN Ở ĐÂY <<<
IMAGE_PATH = "/kaggle/input/datasets/rykanny/img-test-tht2/frame_000001_20261029_085008_260.jpg"

# "bmd_x" | "bmd_s" | "paper_local" | "custom_local"
MODEL_CHOICE = "bmd_x"
PAPER_WEIGHTS_PATH = "/kaggle/input/paper-author-weights/best.pt"
CUSTOM_WEIGHTS_PATH = "/kaggle/input/my-checkpoint/best.pt"

# Parameters: ưu tiên recall trong cảnh đông, nhiều xe máy bị che
IMAGE_SIZE = 960              # thử 640, 960, 1280; 1280 cần GPU mạnh hơn
CONFIDENCE = 0.12             # giảm xuống 0.07 nếu bỏ sót, nhưng tăng false positives
NMS_IOU = 0.65               # ngưỡng NMS trong Ultralytics
MAX_DETECTIONS = 600
TEST_TIME_AUGMENTATION = False  # True chạy chậm hơn, không bảo đảm cải thiện

# Xem cận cảnh khu vực đông xe: normalized x1,y1,x2,y2 (0..1)
CROWDED_REGION = (0.35, 0.38, 0.85, 1.00)
SHOW_CONFIDENCE_ON_BOX = False   # nhiều xe thì tắt cho đỡ rối
SHOW_INDEX_ON_BOX = True
OUTPUT_DIR = "/kaggle/working/traffic_detection_output"


## 2) Kiểm tra môi trường và ảnh

In [ ]:
from pathlib import Path
from collections import Counter
import hashlib, json, re, time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw, ImageFont
import torch
from ultralytics import YOLO
from huggingface_hub import hf_hub_download

image_path = Path(IMAGE_PATH)
if not image_path.is_file():
    example_files = []
    for ext in ("*.jpg", "*.jpeg", "*.png", "*.webp"):
        example_files.extend(list(Path("/kaggle/input").rglob(ext))[:8])
    suggestions = "\n".join(str(x) for x in example_files[:15]) or "(Chưa thấy ảnh nào trong /kaggle/input/)"
    raise FileNotFoundError(
        f"Không tìm thấy IMAGE_PATH: {IMAGE_PATH}\n"
        f"Hãy dùng Add Input ở Kaggle và sửa IMAGE_PATH. Ví dụ file đang có:\n{suggestions}"
    )

output_dir = Path(OUTPUT_DIR)
output_dir.mkdir(parents=True, exist_ok=True)
image = Image.open(image_path).convert("RGB")
W, H = image.size
DEVICE = 0 if torch.cuda.is_available() else "cpu"
print("Image:", image_path, f"| {W} x {H} | GPU:", torch.cuda.get_device_name(0) if DEVICE == 0 else "Không có GPU")
print("Model choice:", MODEL_CHOICE)
if min(W, H) < 400:
    print("⚠️ Ảnh tương đối nhỏ. Tăng imgsz không thể khôi phục chi tiết đã mất. Ưu tiên nguồn camera gốc nếu có.")
plt.figure(figsize=(13, 8))
plt.imshow(image); plt.axis("off"); plt.title("Ảnh gốc"); plt.show()


## 3) Tự tải checkpoint **đã fine-tune** hoặc nạp checkpoint của tác giả

In [ ]:
BMD45_REPO = "iisc-aim/BMD-45"
BMD45_FILES = {
    "bmd_x": "weights/YOLOv12-X/best.pt",  # 356 MB
    "bmd_s": "weights/YOLOv12-S/best.pt",  # 18.9 MB
}

if MODEL_CHOICE in BMD45_FILES:
    filename = BMD45_FILES[MODEL_CHOICE]
    print(f"Đang lấy {BMD45_REPO}/{filename} (weights BMD-45, KHÔNG phải paper Việt Nam)")
    try:
        weights_path = Path(hf_hub_download(repo_id=BMD45_REPO, filename=filename))
    except Exception as exc:
        raise RuntimeError(
            "Không tải được checkpoint Hugging Face. Trên Kaggle hãy bật Internet, hoặc "
            "tải best.pt rồi Add Input, chuyển MODEL_CHOICE='custom_local'."
        ) from exc
elif MODEL_CHOICE == "paper_local":
    weights_path = Path(PAPER_WEIGHTS_PATH)
    if not weights_path.is_file():
        raise FileNotFoundError(
            "Chưa có checkpoint chính thức của paper Việt Nam. "
            f"Đường dẫn không tồn tại: {weights_path}. "
            "Không tự thay bằng yolo11m.pt COCO. Hãy xin best.pt từ tác giả và Add Input."
        )
elif MODEL_CHOICE == "custom_local":
    weights_path = Path(CUSTOM_WEIGHTS_PATH)
    if not weights_path.is_file():
        raise FileNotFoundError(f"Không tìm thấy CUSTOM_WEIGHTS_PATH: {weights_path}")
else:
    raise ValueError(f"MODEL_CHOICE chưa hỗ trợ: {MODEL_CHOICE}")

print("Checkpoint:", weights_path)
print("Size (MB):", round(weights_path.stat().st_size / (1024 * 1024), 2))
print("SHA256 (tính trên file đã tải):", hashlib.sha256(weights_path.read_bytes()).hexdigest())
print("\n⚠️ Checkpoint .pt có thể chứa mã tuần tự hóa Python; chỉ tải từ nguồn bạn tin cậy.")
model = YOLO(str(weights_path))
class_names = {int(k): str(v) for k, v in model.names.items()} if isinstance(model.names, dict) else dict(enumerate(model.names))
print("\nClasses thực tế trong checkpoint:")
for i, name in sorted(class_names.items()):
    print(f"  {i:>2}: {name}")

if MODEL_CHOICE.startswith("bmd"):
    assert any("two" in name.lower() and "wheeler" in name.lower() for name in class_names.values()), (
        "Checkpoint BMD45 không có class Two-wheeler như mong đợi: kiểm tra lại model/source."
    )


## 4) Inference trên đúng ảnh của bạn

In [ ]:
start = time.perf_counter()
res = model.predict(
    source=str(image_path), imgsz=IMAGE_SIZE,
    conf=CONFIDENCE, iou=NMS_IOU, max_det=MAX_DETECTIONS,
    augment=TEST_TIME_AUGMENTATION, device=DEVICE,
    verbose=False, save=False
)[0]
elapsed = time.perf_counter() - start

# Không tự suy đoán class ID: đọc từ chính model.names!
TWO_WHEELER_CLASS_IDS = [
    idx for idx, name in class_names.items()
    if re.search(r"two[ -]?wheeler|motorbike|motorcycle|scooter|moped", name, re.I)
]
# Nếu paper dùng class tiếng Anh khác tên trên, hãy kiểm tra model.names ở Cell 3
rows = []
if res.boxes is not None:
    for idx, box in enumerate(res.boxes):
        cls_id = int(box.cls.item())
        xyxy = [float(x) for x in box.xyxy[0].tolist()]
        confidence = float(box.conf.item())
        x1, y1, x2, y2 = xyxy
        rows.append(dict(
            index=idx+1, class_id=cls_id, class_name=class_names.get(cls_id, str(cls_id)),
            confidence=round(confidence, 5),
            x1=round(x1, 2), y1=round(y1, 2), x2=round(x2, 2), y2=round(y2, 2),
            cx=round((x1+x2)/2, 2), cy=round((y1+y2)/2, 2),
            width=round(x2-x1, 2), height=round(y2-y1, 2),
            is_two_wheeler=(cls_id in TWO_WHEELER_CLASS_IDS)
        ))

df = pd.DataFrame(rows, columns=[
    "index", "class_id", "class_name", "confidence", "x1", "y1", "x2", "y2",
    "cx", "cy", "width", "height", "is_two_wheeler"
])
print(f"Inference hoàn tất sau {elapsed:.2f}s (bao gồm overhead lần đầu) | Tổng {len(df)} bbox")
print(f"Xe 2 bánh: {int(df.is_two_wheeler.sum())} bbox | Class IDs: {TWO_WHEELER_CLASS_IDS}")
print("Lưu ý: nếu paper gán nhãn 'rider' thay cho motorbike, hãy bổ sung vào bộ lọc tên class ở đây.")
print("\nPhân bố dự đoán:")
print(df.class_name.value_counts().to_string() if not df.empty else "Không có detection")
display(df.sort_values("confidence", ascending=False).head(20))


## 5) Vẽ bounding box từng xe hai bánh (màu cam), các phương tiện khác (màu xanh)

In [ ]:
def draw_boxes(pil_image, det_df, two_wheelers_only=False):
    canvas = pil_image.copy()
    drawer = ImageDraw.Draw(canvas)
    thickness = max(2, round(min(pil_image.size) / 300))
    for row in det_df.itertuples(index=False):
        if two_wheelers_only and not row.is_two_wheeler:
            continue
        is_two = bool(row.is_two_wheeler)
        color = (255, 125, 15) if is_two else (25, 210, 155)
        rect = tuple(map(lambda x: int(round(x)), (row.x1, row.y1, row.x2, row.y2)))
        drawer.rectangle(rect, outline=color, width=thickness)
        tag = str(row.index) if SHOW_INDEX_ON_BOX else ""
        if SHOW_CONFIDENCE_ON_BOX:
            tag = (tag + " " if tag else "") + f"{row.confidence:.2f}"
        if tag:
            drawer.text((max(0, rect[0]), max(0, rect[1]-12)), tag, fill=color,
                        stroke_width=1, stroke_fill=(0,0,0))
    return canvas

all_boxes_img = draw_boxes(image, df)
two_wheelers_img = draw_boxes(image, df, two_wheelers_only=True)
fig, axes = plt.subplots(1, 2, figsize=(20, 10))
for ax, im, title in zip(axes, [all_boxes_img, two_wheelers_img],
                         [f"Tất cả phương tiện / đối tượng: {len(df)}", f"Chỉ xe hai bánh: {int(df.is_two_wheeler.sum())}"]):
    ax.imshow(im); ax.axis("off"); ax.set_title(title)
plt.tight_layout(); plt.show()

p1 = output_dir / f"{image_path.stem}_{MODEL_CHOICE}_all_boxes.png"
p2 = output_dir / f"{image_path.stem}_{MODEL_CHOICE}_two_wheelers.png"
all_boxes_img.save(p1); two_wheelers_img.save(p2)
print("Đã lưu:", p1, "\n", p2)


## 6) Zoom khu vực xe máy chen chúc

In [ ]:
x1f, y1f, x2f, y2f = CROWDED_REGION
assert 0 <= x1f < x2f <= 1 and 0 <= y1f < y2f <= 1, "CROWDED_REGION phải là normalized x1,y1,x2,y2"
rect = (int(x1f * W), int(y1f * H), int(x2f * W), int(y2f * H))
fig, axes = plt.subplots(1, 2, figsize=(18, 9))
axes[0].imshow(image.crop(rect)); axes[0].axis("off"); axes[0].set_title("Vùng đông xe: ảnh gốc")
axes[1].imshow(two_wheelers_img.crop(rect)); axes[1].axis("off"); axes[1].set_title("Vùng đông xe: bbox hai bánh")
plt.tight_layout(); plt.show()
zoom_path = output_dir / f"{image_path.stem}_{MODEL_CHOICE}_crowded_zoom.png"
two_wheelers_img.crop(rect).save(zoom_path)


## 7) Thống kê theo ngưỡng confidence và xuất CSV / JSON

In [ ]:
summary = pd.DataFrame([
    {"confidence_at_least": t,
     "all_detections": int((df.confidence >= t).sum()),
     "two_wheelers": int(((df.confidence >= t) & df.is_two_wheeler).sum())}
    for t in [0.07, 0.10, 0.12, 0.15, 0.20, 0.25, 0.35, 0.50]
])
print("Lưu ý: bảng chỉ dùng detections ở CONFIDENCE đã chạy (không phục hồi prediction đã lọc).")
display(summary)

csv_path = output_dir / f"{image_path.stem}_{MODEL_CHOICE}_detections.csv"
summary_path = output_dir / f"{image_path.stem}_{MODEL_CHOICE}_threshold_counts.csv"
meta_path = output_dir / f"{image_path.stem}_{MODEL_CHOICE}_metadata.json"
df.to_csv(csv_path, index=False)
summary.to_csv(summary_path, index=False)
meta_path.write_text(json.dumps({
    "image": str(image_path), "image_size_wh": [W,H], "checkpoint": str(weights_path),
    "weight_source": MODEL_CHOICE, "class_names": class_names,
    "two_wheeler_class_ids": TWO_WHEELER_CLASS_IDS,
    "conf": CONFIDENCE, "iou": NMS_IOU, "imgsz": IMAGE_SIZE,
    "total_predictions": int(len(df)), "two_wheeler_predictions": int(df.is_two_wheeler.sum()),
    "paper_original_weights": MODEL_CHOICE == "paper_local",
    "note": "Paper original weights availability is not verified; paper_local is user supplied. Number of detections is not ground truth."
}, indent=2, ensure_ascii=False), encoding="utf-8")
print("Đã lưu vào", output_dir)
for p in [csv_path, summary_path, meta_path, p1, p2, zoom_path]:
    print("-", p)


## 8) Cách so sánh công bằng (đề xuất)

1. Chạy `MODEL_CHOICE="bmd_x"` với `CONFIDENCE=0.12`, `IMAGE_SIZE=960`.
2. Đổi sang `"bmd_s"` rồi chạy lại cell 3 trở xuống. Hai kết quả lưu theo tên model khác nhau.
3. Nếu nguồn camera là **512×288**, cần cẩn thận vì nhiều xe bị che khuất, và tăng `imgsz` chỉ phóng to chứ không tạo thêm pixel thật. Nếu có frame gốc HD thì dùng frame gốc.
4. Nếu có **ground truth** từng xe, hãy tính precision/recall. Chỉ số "đếm được N xe" tự nó không chứng minh detect tốt.
5. Nếu nhóm tác giả paper Việt Nam cung cấp `best.pt`, Add Input weights vào Kaggle và chuyển `MODEL_CHOICE="paper_local"`, điền `PAPER_WEIGHTS_PATH`. Notebook **không giả mạo checkpoint**.

**Nguồn:** [paper Việt Nam](https://doi.org/10.32604/cmc.2026.078756), [BMD-45 model weights](https://huggingface.co/iisc-aim/BMD-45/tree/main/weights), [BMD-45 paper](https://arxiv.org/abs/2511.02563).